# Dia6b - Fato Vendas Gold (Star Schema)

**Objetivo:** Criar a tabela fato Gold `fat_vendas` em modelo estrela (star schema) com chaves substitutas e foreign keys, criar a Silver intermediária e popular a fato via `MERGE`.

| Item | Valor |
|---|---|
| **Camada** | Silver + Gold (fato) |
| **Entrada** | `{catalog}.bronze.sales`, `{catalog}.gold.{dim_horas,dim_produtos,dim_clientes}` |
| **Saída** | `{catalog}.silver.fat_vendas`, `{catalog}.gold.fat_vendas` |
| **Ordem no pipeline** | Após Dia5-Create-Tables-Indian (que cria as dimensões Gold) |

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Gold — DDL da tabela fato `fat_vendas` (star schema)

Cria a tabela fato com chaves substitutas (SK) para as 4 dimensões e adiciona foreign keys constraint para integridade referencial. `CREATE OR REPLACE TABLE` garante idempotência da DDL.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`gold`.`fat_vendas` ( 
	ven_sk_data_venda BIGINT NOT NULL COMMENT 'SK da data da venda',
	ven_sk_hora SMALLINT NOT NULL COMMENT 'SK hora da venda',
	ven_sk_produto BIGINT NOT NULL COMMENT 'SK do produto',
	ven_sk_cliente BIGINT NOT NULL COMMENT 'SK do cliente',
	ven_id_venda STRING NOT NULL COMMENT 'ID da venda',
	ven_qt_itens SMALLINT NOT NULL COMMENT 'Quantidade de itens da venda',
	ven_vl_unitario FLOAT NOT NULL COMMENT 'Valor unitário da venda',
	ven_vl_total FLOAT NOT NULL COMMENT 'Valor total da venda',
	ven_vl_frete FLOAT NOT NULL COMMENT 'Valor do frete',
	ven_vl_desconto FLOAT NOT NULL COMMENT 'Valor do desconto na venda',
	ven_vl_total_pago FLOAT NOT NULL COMMENT 'Valor total pago na venda',
	ven_dh_atualizacao TIMESTAMP NOT NULL DEFAULT from_utc_timestamp(now(), 'GMT-3') COMMENT 'Data/hora da última atualização da linha',
	    CONSTRAINT pk_fat_vendas PRIMARY KEY ( ven_sk_data_venda, ven_sk_hora, ven_sk_produto, ven_sk_cliente )
 ) USING DELTA
    TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported',
                    'delta.logRetentionDuration'='7 days',
					'delta.autoOptimize.autoCompact'='auto',
                    'spark.databricks.delta.autoCompact.enabled'='true');

COMMENT ON TABLE `${catalog}`.`gold`.`fat_vendas` IS 'A tabela [fat_vendas] contém dados sobre as vendas do Indian E-Commerce Sales.';

ALTER TABLE `${catalog}`.`gold`.`fat_vendas` ADD CONSTRAINT fk_fat_vendas_dim_datas FOREIGN KEY ( ven_sk_data_venda ) REFERENCES `${catalog}`.`gold`.`dim_datas` ( dat_sk_data ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE `${catalog}`.`gold`.`fat_vendas` ADD CONSTRAINT fk_fat_vendas_dim_horas FOREIGN KEY ( ven_sk_hora ) REFERENCES `${catalog}`.`gold`.`dim_horas` ( hor_sk_hora ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE `${catalog}`.`gold`.`fat_vendas` ADD CONSTRAINT fk_fat_vendas_dim_produtos FOREIGN KEY ( ven_sk_produto ) REFERENCES `${catalog}`.`gold`.`dim_produtos` ( pro_sk_produto ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE `${catalog}`.`gold`.`fat_vendas` ADD CONSTRAINT fk_fat_vendas_dim_clientes FOREIGN KEY ( ven_sk_cliente ) REFERENCES `${catalog}`.`gold`.`dim_clientes` ( cli_sk_cliente ) ON DELETE NO ACTION ON UPDATE NO ACTION;

## Silver — Limpeza e preparação da fato

Cria a Silver `fat_vendas` a partir de `bronze.sales`, filtrando apenas pedidos `Delivered` com `Total_Amount > 0`. Converte IDs de negócio para o formato que permitirá o JOIN com as dimensões Gold.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`silver`.`fat_vendas` AS
  SELECT
      DATE_FORMAT(Order_Date, 'yyyyMMdd') AS ven_sk_data_venda,
      REPLACE(LEFT(Order_Time,5),':','') AS ven_id_hora,
      TRIM(Product_ID) AS ven_id_produto,
      TRIM(Customer_ID) AS ven_id_cliente,
      TRIM(Order_ID) AS ven_id_venda,
      Quantity AS ven_qt_itens,
      Unit_Price AS ven_vl_unitario,
      Order_Value AS ven_vl_total,
      Shipping_Cost AS ven_vl_frete,
      Coupon_Discount AS ven_vl_desconto,
      Total_Amount AS ven_vl_total_pago,
      from_utc_timestamp(now(), 'GMT-3') AS ven_dh_atualizacao
  FROM `${catalog}`.`bronze`.`sales`
  WHERE 1=1
    AND Order_Status = 'Delivered'
    AND Total_Amount > 0.0;

## MERGE — População da fato Gold com chaves substitutas

Faz `MERGE INTO` da Silver para a Gold, trocando IDs de negócio (texto) pelas chaves substitutas numéricas (SK) das dimensões via `LEFT JOIN`. `MERGE` é idempotente: atualiza vendas existentes e insere novas.

In [0]:
%sql
MERGE INTO `${catalog}`.`gold`.`fat_vendas` AS target
    USING (
        SELECT
            fat.ven_sk_data_venda,
            hor.hor_sk_hora AS ven_sk_hora,
            pro.pro_sk_produto AS ven_sk_produto,
            cli.cli_sk_cliente AS ven_sk_cliente,
            fat.ven_id_venda,
            fat.ven_qt_itens,
            fat.ven_vl_unitario,
            fat.ven_vl_total,
            fat.ven_vl_frete,
            fat.ven_vl_desconto,
            fat.ven_vl_total_pago
        FROM
            `${catalog}`.`silver`.`fat_vendas` AS fat
                LEFT JOIN `${catalog}`.`gold`.`dim_horas` AS hor
                    ON (fat.ven_id_hora = hor.hor_id_hora)
                LEFT JOIN `${catalog}`.`gold`.`dim_produtos` AS pro
                    ON (fat.ven_id_produto = pro.pro_id_produto)
                LEFT JOIN `${catalog}`.`gold`.`dim_clientes` AS cli
                    ON (fat.ven_id_cliente = cli.cli_id_cliente)
    ) AS source
        ON (target.ven_sk_data_venda = source.ven_sk_data_venda AND target.ven_sk_hora = source.ven_sk_hora
            AND target.ven_sk_produto = source.ven_sk_produto AND target.ven_sk_cliente = source.ven_sk_cliente
            AND target.ven_id_venda = source.ven_id_venda)
WHEN MATCHED THEN
    UPDATE SET
        ven_qt_itens        = source.ven_qt_itens,
        ven_vl_unitario     = source.ven_vl_unitario,
        ven_vl_total        = source.ven_vl_total,
        ven_vl_frete        = source.ven_vl_frete,
        ven_vl_desconto     = source.ven_vl_desconto,
        ven_vl_total_pago   = source.ven_vl_total_pago,
        ven_dh_atualizacao  = from_utc_timestamp(now(), 'GMT-3')
WHEN NOT MATCHED THEN
    INSERT (ven_sk_data_venda, ven_sk_hora, ven_sk_produto, ven_sk_cliente, ven_id_venda, ven_qt_itens, ven_vl_unitario, ven_vl_total, ven_vl_frete, ven_vl_desconto, ven_vl_total_pago)
    VALUES (source.ven_sk_data_venda, source.ven_sk_hora, source.ven_sk_produto, source.ven_sk_cliente, source.ven_id_venda, source.ven_qt_itens, source.ven_vl_unitario, source.ven_vl_total, source.ven_vl_frete, source.ven_vl_desconto, source.ven_vl_total_pago);